# Línea base y respuestas de Qwen3-1.7B (IIC-3694)

Este cuaderno **no entrena** el modelo: lo evalúa con el harness y guarda sus respuestas. Corre el modelo objetivo, Qwen3-1.7B con thinking, en dos tandas:

1. **Línea base sobre `eval`:** qué fracción de tareas resuelve, con pass@1 (al primer intento) y pass@5 (al menos uno de 5), por dificultad y condición. Es su nota real y la evidencia de calibración.
2. **Respuestas sobre `train`:** el verificador las separa en trazas SFT (correctas) y pares DPO (una correcta y una incorrecta del mismo prompt). Las trazas de este modelo son provisorias: las definitivas vienen de Qwen3-8B, en el clúster.

Necesita una GPU T4 y el zip del proyecto en Drive (ver `docs/guia_colab.md`). Ejecuta *Entorno de ejecución > Ejecutar todo*. Si Colab se desconecta, vuelve a ejecutar todo: cada tanda continúa donde quedó, sin repetir muestras.

## 1. Verificar la GPU

In [ ]:
import torch

if torch.cuda.is_available():
    GPU = torch.cuda.get_device_name(0)
    memoria = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"GPU detectada: {GPU} ({memoria:.1f} GiB)")
else:
    GPU = None
    print("=" * 70)
    print("ATENCION: NO HAY GPU. En CPU esto tardaria muchas horas.")
    print("Activa la GPU: Entorno de ejecucion > Cambiar tipo de entorno > T4.")
    print("Luego vuelve a ejecutar el cuaderno desde esta celda.")
    print("=" * 70)

## 2. Montar Google Drive

Colab pedirá permiso para acceder a tu Drive. Todos los resultados quedan en `MyDrive/inventory_harness/`.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/inventory_harness")
BASE.mkdir(parents=True, exist_ok=True)
RESULTADOS = BASE / "results"
RESULTADOS.mkdir(exist_ok=True)
ZIP_CODIGO = BASE / "inventory_harness_colab.zip"

if ZIP_CODIGO.exists():
    print("Zip encontrado:", ZIP_CODIGO)
else:
    print(f"FALTA EL ZIP. Sube inventory_harness_colab.zip a {BASE} y repite esta celda.")

## 3. Instalar dependencias y descomprimir el proyecto

No se reinstala `torch`: se usa el que trae Colab, compilado para su GPU. El código se descomprime en el disco local, cada vez desde cero, para no mezclar versiones. Al final corren las pruebas del núcleo, que deben pasar (no usan GPU).

In [ ]:
!pip install -q "transformers>=4.51" accelerate pytest bitsandbytes

In [ ]:
import os
import shutil
import sys
import zipfile

CODIGO = Path("/content/inventory_harness")
shutil.rmtree(CODIGO, ignore_errors=True)
with zipfile.ZipFile(ZIP_CODIGO) as archivo:
    archivo.extractall(CODIGO)
sys.path.insert(0, str(CODIGO / "src"))
os.chdir(CODIGO)
print("Proyecto en", CODIGO)

In [ ]:
!python -m pytest -q tests/test_solver.py tests/test_numbers_es.py tests/test_verifier.py tests/test_generator.py

## 4. Configuración

**Esta es la única celda que puedes editar.** Los valores por omisión suman 1.260 rollouts. La celda imprime una estimación de tiempo con lo medido en la exploración; con el tope de 4.096 tokens puede tardar más. Si no te alcanza la sesión, baja primero `TRAZAS_POR_CELDA`.

In [ ]:
MODELO = "Qwen/Qwen3-1.7B"
BATCH_SIZE = None  # None usa 16; si falta memoria, el lote se reduce solo.

# Línea base sobre eval: instancias por celda (hay 9 celdas) y muestras por instancia.
LINEA_BASE_POR_CELDA = 10
LINEA_BASE_MUESTRAS = 5

# Respuestas sobre train, para SFT y DPO.
TRAZAS_POR_CELDA = 10
TRAZAS_MUESTRAS = 2

# ------------------- No hace falta editar desde aquí -------------------
import json
import subprocess

DIR_LINEA_BASE = RESULTADOS / "linea_base_qwen17b"
DIR_TRAZAS = RESULTADOS / "trazas_qwen17b"
DIR_DATOS = RESULTADOS / "datos_derivados_qwen17b"


def correr(*argumentos):
    """Ejecuta un script del proyecto y muestra su salida mientras avanza."""
    comando = [sys.executable, "-u", *map(str, argumentos)]
    proceso = subprocess.Popen(
        comando, cwd=CODIGO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for linea in proceso.stdout:
        print(linea, end="")
    return proceso.wait()


def extra_lote():
    return [] if BATCH_SIZE is None else ["--batch-size", BATCH_SIZE]


rollouts = 9 * 2 * (LINEA_BASE_POR_CELDA * LINEA_BASE_MUESTRAS + TRAZAS_POR_CELDA * TRAZAS_MUESTRAS)
# La exploración midió 14,9 s por rollout con thinking en una T4, con lotes de 16.
print(f"{rollouts} rollouts, unas {rollouts * 14.9 / 3600:.1f} h en una T4 (estimación).")

## 5. Prueba de humo

Corre 9 instancias, una por celda, con 1 muestra. Revisa el rollout que imprime: el modelo debería pedir datos con `get_inventory` y, tras los resultados, terminar con un bloque `<answer>`.

In [ ]:
HUMO = Path("/content/humo")
shutil.rmtree(HUMO, ignore_errors=True)
codigo = correr(
    "scripts/run_pilot.py", "--stage", "exploracion", "--models", MODELO, "--thinking", "on",
    "--limit", 9, "--n-samples", 1, "--out", HUMO, *extra_lote(),
)
print("Código de salida:", codigo)

archivos = sorted(HUMO.glob("rollouts_*.jsonl"))
if not archivos:
    print("No hay rollouts. Lee", HUMO / "manifest.json")
    print((HUMO / "manifest.json").read_text(encoding="utf-8"))
for archivo in archivos:
    filas = [json.loads(l) for l in archivo.read_text(encoding="utf-8").splitlines() if l.strip()]
    fila = next((f for f in filas if f["condition"] == "tools"), filas[0])
    print("#" * 30, fila["model"], "#" * 30)
    print("=" * 30, "PROMPT", "=" * 30)
    print(fila["prompt"])
    for mensaje in fila["messages"]:
        print("=" * 30, mensaje["role"].upper(), "=" * 30)
        print(mensaje["content"])
    print("=" * 30, "VEREDICTO", "=" * 30)
    print(json.dumps(fila["verdict"], ensure_ascii=False, indent=2))
    print("Uso:", fila["usage"])
    print(f"Aciertos en el humo: {sum(f['verdict']['reward'] == 1.0 for f in filas)}/{len(filas)}")


## 6. Línea base sobre `eval` (nota real)

Usa la semilla 2026 y el split `eval`. Al terminar, `analyze_results.py` escribe `report.md` con pass@1 y pass@5 por dificultad y condición, y la celda siguiente lo muestra.

In [ ]:
codigo = correr(
    "scripts/run_pilot.py", "--stage", "piloto", "--models", MODELO, "--thinking", "on",
    "--n-per-cell", LINEA_BASE_POR_CELDA, "--n-samples", LINEA_BASE_MUESTRAS,
    "--out", DIR_LINEA_BASE, *extra_lote(),
)
print("Código de salida:", codigo)
correr("scripts/analyze_results.py", DIR_LINEA_BASE)

In [ ]:
from IPython.display import Markdown, display

display(Markdown((DIR_LINEA_BASE / "report.md").read_text(encoding="utf-8")))

## 7. Respuestas sobre `train` y datos SFT/DPO

Las instancias de `train` salen de la misma generación que las de `eval` y no repiten ninguna. Después, `build_training_data.py` pasa cada respuesta por el verificador y escribe `sft.jsonl`, `dpo.jsonl` y `manifest.json`, con los conteos por celda.

In [ ]:
codigo = correr(
    "scripts/run_pilot.py", "--stage", "trazas", "--models", MODELO, "--thinking", "on",
    "--n-per-cell", TRAZAS_POR_CELDA, "--n-samples", TRAZAS_MUESTRAS,
    "--out", DIR_TRAZAS, *extra_lote(),
)
print("Código de salida:", codigo)
correr("scripts/build_training_data.py", DIR_TRAZAS, "--out", DIR_DATOS)

## 8. Guardar resultados

Crea **`MyDrive/inventory_harness/resultados_qwen17b.zip`** con los rollouts, los informes y los datos SFT/DPO. Sirve también con una corrida incompleta.

In [ ]:
import datetime

ZIP_RESULTADOS = BASE / "resultados_qwen17b.zip"
local = Path("/content/resultados_qwen17b.zip")
etapas = [d for d in (DIR_LINEA_BASE, DIR_TRAZAS) if (d / "manifest.json").exists()]
carpetas = etapas + ([DIR_DATOS] if DIR_DATOS.exists() else [])
if not carpetas:
    print("Todavía no hay resultados.")
else:
    for carpeta in etapas:
        correr("scripts/analyze_results.py", carpeta)
    with zipfile.ZipFile(local, "w", zipfile.ZIP_DEFLATED) as archivo:
        for carpeta in carpetas:
            for ruta in sorted(carpeta.iterdir()):
                if ruta.is_file() and not ruta.name.endswith(".tmp"):
                    archivo.write(ruta, f"{carpeta.name}/{ruta.name}")
    shutil.copy(local, ZIP_RESULTADOS)
    print(datetime.datetime.now().isoformat(timespec="seconds"))
    print(f"LISTO ({local.stat().st_size / 2**20:.1f} MiB):")
    print(ZIP_RESULTADOS)

Si prefieres descargarlo directamente a tu computador, ejecuta esta celda. La copia en Drive ya está hecha.

In [ ]:
from google.colab import files

files.download(str(local))